In [0]:
import random
import time
from datetime import datetime, timedelta

PRODUCTS = {
    "P001": 65000,
    "P002": 25000,
    "P003": 18000,
    "P004": 12000,
    "P005": 35000,
    "P006": 1500,
    "P007": 700,
    "P008": 2500
}

def generate_orders():

    # Random number of records every run
    num_records = random.randint(10000, 20000)

    data = []

    start_date = datetime(2026, 1, 1)
    end_date = datetime(2026, 6, 30)
    total_days = (end_date - start_date).days

    # Unique base for this run (milliseconds)
    order_base = int(time.time() * 1000)

    for i in range(num_records):

        product_id = random.choice(list(PRODUCTS.keys()))
        quantity = random.randint(1, 5)
        unit_price = PRODUCTS[product_id]

        order_date = (
            start_date + timedelta(days=random.randint(0, total_days))
        ).strftime("%Y-%m-%d")

        data.append((
            f"{order_base}{i:05d}",      # Example: O175164256812300001
            f"{random.randint(1001, 1200)}",
            product_id,
            quantity,
            quantity * unit_price,
            order_date
        ))

    columns = [
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "price",
        "order_date"
    ]

    return spark.createDataFrame(data, columns)

In [0]:
# Generate random data
df_orders = generate_orders()

# Write to a Volume
from datetime import datetime
from pyspark.sql.functions import *

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Temporary folder
temp_path = f"/Volumes/associate_assignment/default/raw/order_volume/temp/orders_{timestamp}"

# Final file path
final_file = f"/Volumes/associate_assignment/default/raw/order_volume/landing/orders_{timestamp}.csv"

# Write DataFrame (creates one part file)
(
    df_orders
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(temp_path)
)

# Find the generated CSV file
files = dbutils.fs.ls(temp_path)

for file in files:
    if file.name.endswith(".csv"):
        dbutils.fs.mv(file.path, final_file)
        break

# Remove temporary folder
dbutils.fs.rm(temp_path, recurse=True)

print(f"File created: {final_file}")